In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import requests


In [ ]:
ano = 2022

In [ ]:
data_dir = Path("data")
INDIR = Path(f"../data/data_model/{ano}")
OUTDIR_IMG = Path(f"../report/img/{ano}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [ ]:
city_file = INDIR / f"ANALISE_NOTAS_ENEM_MUNICIPIOS_BRASIL_CLUSTERS_{ano}.csv"
df_city = pd.read_csv(city_file, sep=",")

In [ ]:
df_city.head()

In [ ]:
city_clusters = {
    0: df_city[df_city['CLUSTER'] == 0].copy().reset_index(drop=True),
    1: df_city[df_city['CLUSTER'] == 1].copy().reset_index(drop=True),
    2: df_city[df_city['CLUSTER'] == 2].copy().reset_index(drop=True)
}

In [ ]:
income_col = "FAMILY_INCOME_SM_AVG"

score_columns = ['NATURAL_SCIENCES_SCORE_AVG', 'HUMANITIES_SCORE_AVG', 'LANGUAGES_SCORE_AVG', 'MATH_SCORE_AVG', 'ESSAY_SCORE_AVG']

subject_names = {
    'CN': 'Natural Sciences and its Technologies',
    'MT': 'Mathematics and its Technologies',
    'CH': 'Humanities and its Technologies',
    'LC': 'Languages, Codes and its Technologies',
    'REDACAO': 'Essay'
}

score_cols = [c for c in score_columns if c in df_city.columns]
cluster_colors = {0: "#ff0e0e", 1: "#1f77b4", 2: "#2ca02c"}

performance_labels = {0: "Low", 1: "Intermediate", 2: "High"}
performance_colors = {
    "Low": cluster_colors[0],
    "Intermediate": cluster_colors[1],
    "High": cluster_colors[2],
}

In [ ]:
colunas_plot = score_cols.copy()
if "OVERALL_SCORE_AVG" in df_city.columns and "OVERALL_SCORE_AVG" not in colunas_plot:
    colunas_plot.append("OVERALL_SCORE_AVG")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, coluna in enumerate(colunas_plot):
    ax = axes[i]
    for cluster in sorted(df_city["CLUSTER"].unique()):
        desempenho = performance_labels.get(cluster, f"Cluster {cluster}")
        ax.hist(
            df_city.loc[df_city["CLUSTER"] == cluster, coluna],
            bins=20,
            alpha=0.55,
            label=desempenho,
            color=performance_colors.get(desempenho),
            density=True
        )

    if coluna == "OVERALL_SCORE_AVG":
        titulo = "Média Geral"
    else:
        area_sigla = coluna.replace("NOTA_", "").replace("_MEDIA", "")
        titulo = subject_names.get(area_sigla, area_sigla)

    ax.set_title(titulo)
    ax.set_xlabel("Nota")
    ax.set_ylabel("Frequência")
    ax.legend()

for j in range(len(colunas_plot), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Distribuição das Médias de Notas por Cluster", fontsize=14)
plt.show()

In [ ]:
colunas_plot = score_cols.copy()
if "OVERALL_SCORE_AVG" in df_city.columns and "OVERALL_SCORE_AVG" not in colunas_plot:
    colunas_plot.append("OVERALL_SCORE_AVG")

titulos = []
for c in colunas_plot:
    if c == "OVERALL_SCORE_AVG":
        area_nome = "Média Geral"
    else:
        area_sigla = c.replace("NOTA_", "").replace("_MEDIA", "")
        area_nome = subject_names.get(area_sigla, area_sigla)

    d = df_city[[income_col, c]].dropna()
    corr = d[income_col].corr(d[c])
    titulos.append(f"{area_nome}<br>(Correlação: {corr:.3f})")

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=titulos,
    horizontal_spacing=0.08,
    vertical_spacing=0.25
)

for i, coluna in enumerate(colunas_plot):
    row = i // 3 + 1
    col = i % 3 + 1

    for cluster in sorted(df_city["CLUSTER"].unique()):
        desempenho = performance_labels.get(cluster, f"Cluster {cluster}")
        dados = df_city[df_city["CLUSTER"] == cluster]
        dados_validos = dados[[income_col, coluna, "CITY", "STATE", "NUM_PARTICIPANTS"]].dropna()

        fig.add_trace(
            go.Scatter(
                x=dados_validos[income_col],
                y=dados_validos[coluna],
                mode="markers",
                marker=dict(size=6, color=performance_colors.get(desempenho)),
                opacity=0.7,
                name=desempenho,
                showlegend=(i == 0),
                customdata=dados_validos[["CITY", "STATE", "NUM_PARTICIPANTS"]],
                hovertemplate=(
                    "Município: %{customdata[0]} (%{customdata[1]})<br>"
                    "Renda: %{x:.2f}<br>"
                    "Nota: %{y:.2f}<br>"
                    "Participantes: %{customdata[2]}<br>"
                    f"Performance: {desempenho}"
                    "<extra></extra>"
                )
            ),
            row=row,
            col=col
        )

    fig.update_xaxes(title_text="Renda Familiar Média (SM)", row=row, col=col)
    fig.update_yaxes(title_text="Nota Média", row=row, col=col)

fig.update_layout(
    title=f"Relação entre Renda Familiar Média e Notas por Performance (ENEM {ano})",
    height=800,
    width=1200,
    template="plotly_white",
    legend_title_text="Performance" 
 )

fig.show()

In [ ]:
url = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"
geojson = requests.get(url).json()

In [ ]:
df_map = df_city.copy()

df_map["CITY_CODE"] = df_map["CITY_CODE"].astype(str)
df_map["PERFORMANCE"] = pd.Categorical(
    df_map["CLUSTER"].map(performance_labels),
    ordered=True,
    )

In [ ]:
fig = px.choropleth(
    df_map,
    geojson=geojson,
    locations="CITY_CODE",
    featureidkey="properties.id",
    color="PERFORMANCE",
    color_discrete_map=performance_colors,
    title=f"Performance por município (Brasil - ENEM {ano})",
    custom_data=[
        "CITY",
        "STATE",
        "PERFORMANCE",
        "OVERALL_SCORE_AVG",
        "FAMILY_INCOME_SM_AVG",
        "NUM_PARTICIPANTS"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "Município: %{customdata[0]}<br>"
        "UF: %{customdata[1]}<br>"
        "Performance: %{customdata[2]}<br>"
        "Média Geral: %{customdata[3]:.2f}<br>"
        "Renda média: %{customdata[4]:.2f}<br>"
        "Participantes: %{customdata[5]}<br>"
        "<extra></extra>"
    )
)

fig.show()

In [ ]:
selected_state = "RS"

In [ ]:
df_map_state = df_map.copy()
df_map_state = df_map[df_map["STATE"] == selected_state]

df_map_state["CITY_CODE"] = df_map_state["CITY_CODE"].astype(str).str.zfill(7)

fig = px.choropleth(
    df_map_state,
    geojson=geojson,
    locations="CITY_CODE",
    featureidkey="properties.id",
    color="PERFORMANCE",
    color_discrete_map=performance_colors,
    title=f"Performance por município ({selected_state} - ENEM {ano})",
    custom_data=[
        "CITY",
        "STATE",
        "PERFORMANCE",
        "OVERALL_SCORE_AVG",
        "FAMILY_INCOME_SM_AVG",
        "NUM_PARTICIPANTS"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "Município: %{customdata[0]}<br>"
        "UF: %{customdata[1]}<br>"
        "Performance: %{customdata[2]}<br>"
        "Média Geral: %{customdata[3]:.2f}<br>"
        "Renda média: %{customdata[4]:.2f}<br>"
        "Participantes: %{customdata[5]}<br>"
        "<extra></extra>"
    )
)

fig.show()